In [3]:
# Install required libraries
!pip install -q sentence-transformers transformers torch torchvision numpy pillow chromadb openai spacy clip pyngrok


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#**Semantic Chnuking**

In [5]:
import json
import spacy

def semantic_chunking(text, chunk_size=450, overlap=200):
    """
    Splits text into semantically meaningful chunks using sentence boundaries.
    """
    nlp = spacy.load("en_core_web_sm")  # Load small spaCy model for efficient processing
    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents]  # Extract sentences

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence)

        if current_length + sentence_length > chunk_size and current_chunk:
            # Store current chunk
            chunks.append(" ".join(current_chunk))

            # Create overlap with previous chunk
            overlap_sentences = current_chunk[-overlap:] if overlap < len(current_chunk) else current_chunk[:]
            current_chunk = overlap_sentences[:]
            current_length = sum(len(s) for s in current_chunk)

        current_chunk.append(sentence)
        current_length += sentence_length

    # Add the final chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# Load the cleaned JSON file
file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/cleaned_winning_models.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Process each entry and apply semantic chunking
final_chunked_data = []
for entry in data:
    model_name = entry.get("model_name", "")
    description = entry.get("description", "")
    system_benefits = entry.get("system_benefits", "")
    image_path = entry.get("image_path", "")
    image_description = entry.get("image_description", "")
    url = entry.get("url", "")
    category = entry.get("category", "")
    additional_metadata = {key: value for key, value in entry.items() if key not in ["model_name", "description", "system_benefits", "image_path", "image_description", "url", "category"]}

    # Concatenating relevant fields
    full_text = f"{model_name}. {description} {system_benefits}"

    # Apply semantic chunking
    text_chunks = semantic_chunking(full_text, chunk_size=450, overlap=200)

    # Create chunked entries while keeping image_path and metadata associated
    for idx, chunk in enumerate(text_chunks):
        chunk_entry = {
            "model_name": model_name,
            "chunk_index": idx,
            "chunk_text": chunk,
            "image_path": image_path,
            "image_description": image_description,
            "url": url,
            "category": category
        }
        chunk_entry.update(additional_metadata)  # Add extra metadata
        final_chunked_data.append(chunk_entry)

# Save the final chunked JSON file
final_chunked_file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/semantic_chunked_models.json"
with open(final_chunked_file_path, "w", encoding="utf-8") as file:
    json.dump(final_chunked_data, file, indent=4, ensure_ascii=False)

# Provide the final chunked file path
print(f"Chunked data saved at: {final_chunked_file_path}")



Chunked data saved at: /content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/semantic_chunked_models.json


In [6]:
import json

# Path to your JSON file
file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/semantic_chunked_models.json"

# Load JSON
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Count total records
print(f"Total Records in JSON: {len(data)}")


Total Records in JSON: 257


# **Implementing CLIP Image + Text Embeddings with Weighted Fusion**
**Since we want to combine image and text embeddings efficiently, we’ll use weighted fusion, giving 70% importance to text and 30% to image**

In [11]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def split_text_into_chunks(text, max_length=77):
    """Splits text into chunks of max_length tokens (77 for CLIP)."""
    words = text.split()  # Split text into words
    chunks = []

    for i in range(0, len(words), max_length):
        chunk = " ".join(words[i:i+max_length])
        chunks.append(chunk)

    return chunks


def generate_clip_text_embedding(text):
    """Generate CLIP embedding for text, handling long inputs by chunking."""
    text_chunks = split_text_into_chunks(text)  # Split text if needed
    embeddings = []

    for chunk in text_chunks:
        inputs = clip_processor(text=[chunk], return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)
        with torch.no_grad():
            chunk_embedding = clip_model.get_text_features(**inputs)
        embeddings.append(chunk_embedding.squeeze().cpu().numpy())

    # Return the average of chunk embeddings (or try another fusion method)
    return np.mean(embeddings, axis=0)



In [12]:
def generate_clip_image_embedding(image_path):
    """Generate CLIP embedding for an image."""
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = clip_processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_embedding = clip_model.get_image_features(**inputs)
        return image_embedding.squeeze().cpu().numpy()
    except Exception as e:
        print(f"⚠️ Error processing image {image_path}: {e}")
        return None  # If image fails, return None


In [13]:
def weighted_fusion(text_embedding, image_embedding, text_weight=0.7, image_weight=0.3):
    """Fuse embeddings using weighted averaging."""
    if image_embedding is None:  # If no image available, use only text
        return text_embedding
    return (text_embedding * text_weight) + (image_embedding * image_weight)


In [14]:
import json
import numpy as np

# Define file paths
chunked_json_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/semantic_chunked_models.json"
embedded_json_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/final_embedded_data.json"  # Where embeddings will be saved

# Load chunked JSON file
with open(chunked_json_path, "r", encoding="utf-8") as file:
    chunked_data = json.load(file)

# Initialize list to store embedded records
embedded_data = []

# Process each chunk and generate embeddings
for entry in chunked_data:
    model_name = entry.get("model_name", "")
    chunk_index = entry.get("chunk_index", 0)
    chunk_text = entry.get("chunk_text", "")
    image_path = entry.get("image_path", "")

    # Generate CLIP embeddings
    text_embedding = generate_clip_text_embedding(chunk_text)
    image_embedding = generate_clip_image_embedding(image_path) if image_path else None

    # Apply weighted fusion (70% text, 30% image)
    final_embedding = weighted_fusion(text_embedding, image_embedding)

    # Prepare embedded record
    embedded_entry = {
        "model_name": model_name,
        "chunk_index": chunk_index,
        "chunk_text": chunk_text,
        "image_path": image_path,
        "embedding": final_embedding.tolist()  # Convert to JSON serializable format
    }

    # Store in list
    embedded_data.append(embedded_entry)

# Save embeddings to a JSON file
with open(embedded_json_path, "w", encoding="utf-8") as file:
    json.dump(embedded_data, file, indent=4, ensure_ascii=False)

print(f"✅ CLIP Image + Text embeddings saved successfully to {embedded_json_path}")


✅ CLIP Image + Text embeddings saved successfully to /content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/final_embedded_data.json


In [15]:
import json

# Path to your JSON file
file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/final_embedded_data.json"

# Load JSON
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Count total records
print(f"Total Records in JSON: {len(data)}")

Total Records in JSON: 257


In [16]:
unique_texts = set(entry["chunk_text"] for entry in data)

print(f"Total Unique Chunk Texts: {len(unique_texts)}")


Total Unique Chunk Texts: 257


In [17]:
unique_models = set(entry["model_name"] for entry in data)

print(f"Total Unique Model Names: {len(unique_models)}")
print("Unique Models:", unique_models)


Total Unique Model Names: 48
Unique Models: {'Motor Generator System', 'Instrument Panel for Light Electric Vehicles', 'Intelligent Camera Solution', 'Level 2 EV Charger', 'Communication Gateway & Integrated DVR/DMS System Solution', 'xEV Inverter with Inductive Position Sensor (IPS)', 'Standard EV Battery Management System', 'Low-Cost Digital Instrument Cluster', 'Connected Android-Based Vehicle Instrument Cluster', 'High-End Cockpit & Infotainment Solution', 'Wireless Telematic Unit for Vehicle Connectivity', 'ADAS Front Camera Solution', 'Bi-directional GaN DC/DC Converter for 12V/48V EV/HEV Systems', 'Sunroof Controller', 'Automotive Camera Solution with AHL Control Channel', 'Vehicle Control Unit', 'Cost-Effective Automotive Haptic Touch Key Module for Enhanced Driver Safety', 'Full Graphics Cluster & Cockpit Solution', 'Automotive Monitoring Function Extension', '48V Mobility Platform', 'Smart Bicycle Tail Light & Alarm System', '48V/3kW Motor Control for 2/3 Wheelers', 'Advanced

In [18]:
unique_chunks = set((entry["chunk_text"], entry["chunk_index"]) for entry in data)

print(f"Total Unique Chunks: {len(unique_chunks)}")


Total Unique Chunks: 257


In [20]:
unique_embeddings = set(tuple(entry["embedding"]) for entry in data)

print(f"Total Unique Embeddings: {len(unique_embeddings)}")


Total Unique Embeddings: 235


In [57]:
import json

# Path to the embedded JSON file
embedded_json_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/final_embedded_data.json"

# Load JSON data
with open(embedded_json_path, "r", encoding="utf-8") as file:
    embedded_data = json.load(file)

print(f"✅ Loaded {len(embedded_data)} records from the embedded JSON file.")


✅ Loaded 257 records from the embedded JSON file.


In [58]:
# Track unique embeddings and IDs
unique_embeddings = set()
filtered_data = []

for entry in embedded_data:
    embedding_tuple = tuple(entry["embedding"])  # Convert list to tuple for deduplication

    if embedding_tuple not in unique_embeddings:
        unique_embeddings.add(embedding_tuple)
        filtered_data.append(entry)  # Store only unique entries

print(f"✅ Unique records after filtering: {len(filtered_data)}")


✅ Unique records after filtering: 235


In [66]:
import chromadb

# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_chromadb_store")

# Get or create collection with cosine similarity
image_text_collection = chroma_client.get_or_create_collection(
    name="rag_clip_image_text",
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity for retrieval
)

print("✅ ChromaDB collection initialized with cosine similarity.")


✅ ChromaDB collection initialized with cosine similarity.


In [67]:
import chromadb

# Load the collection
image_text_collection = chroma_client.get_or_create_collection(name="rag_clip_image_text")

# Verify if data exists
stored_records = image_text_collection.get()
print(f"✅ Total Records before inserting into ChromaDB: {len(stored_records['ids'])}")


✅ Total Records before inserting into ChromaDB: 0


#**Inserting records into Chromadb**

In [68]:

import chromadb


# Insert unique embeddings into ChromaDB
inserted_count = 0
for entry in filtered_data: #unique filtered embeedings
    try:
        chroma_id = f"{entry['model_name']}_{entry['chunk_index']}"

        image_text_collection.add(
            ids=[chroma_id],
            embeddings=[entry["embedding"]],
            metadatas=[{
                "model_name": entry["model_name"],
                "chunk_index": entry["chunk_index"],
                "chunk_text": entry["chunk_text"],
                "image_path": entry["image_path"]
            }]
        )
        inserted_count += 1

    except Exception as e:
        print(f"⚠️ Failed to insert {chroma_id}: {e}")

print(f"✅ Successfully inserted {inserted_count} unique embeddings into ChromaDB!")


✅ Successfully inserted 235 unique embeddings into ChromaDB!


In [74]:
import chromadb

# Load the collection
image_text_collection = chroma_client.get_or_create_collection(name="rag_clip_image_text")

# Verify if data exists
stored_records = image_text_collection.get()
print(f"✅ Total Records after inserting into ChromaDB: {len(stored_records['ids'])}")

✅ Total Records after inserting into ChromaDB: 235


#**Text input based chromadb search**

In [70]:
from transformers import CLIPProcessor, CLIPModel
import torch

# Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def generate_clip_text_embedding(query_text):
    """Generate CLIP embedding for a text query."""
    inputs = clip_processor(text=[query_text], return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)
    with torch.no_grad():
        text_embedding = clip_model.get_text_features(**inputs)
    return text_embedding.squeeze().cpu().numpy()

# Define a search query
query_text = "How does an ADAS system work?"

# Generate text embedding
query_embedding = generate_clip_text_embedding(query_text)

# Perform semantic search in ChromaDB
search_results = image_text_collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5  # Retrieve top 5 results
)

# Display results with cosine similarity
for i, (result, score) in enumerate(zip(search_results["metadatas"][0], search_results["distances"][0])):
    similarity = 1 - score  # ChromaDB returns L2 distance, convert to cosine similarity
    print(f"\n🔍 **Result {i+1}** (Similarity: {similarity:.4f})")
    print(f"📌 Model: {result['model_name']}")
    print(f"📜 Chunk: {result['chunk_text']}")
    print(f"🖼️ Image Path: {result['image_path']}")



🔍 **Result 1** (Similarity: 0.8002)
📌 Model: Cost-Effective Automotive Haptic Touch Key Module for Enhanced Driver Safety
📜 Chunk: Cost-Effective Automotive Haptic Touch Key Module for Enhanced Driver Safety. This is about application Cost-Effective Automotive Haptic Touch Key Module for Enhanced Driver Safety: As vehicle interiors become more technologically advanced, there is a growing trend toward adopting capacitive touch buttons and interfaces. To ensure driver safety, these interfaces require haptic feedback, allowing drivers to confirm their actions without taking their eyes off the road. This cost-effective automotive touch key module integrates haptic feedback into any control switch, enhancing user experience and promoting vehicle safety.
🖼️ Image Path: /content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/diagrams/Cost-Effective_Automotive_Haptic_Touch_Key_Module_for_Enhanced_Driver_Safety.png

🔍 **Result 2** (Similarity: 0.7910)
📌 Model: Con

#**Image input based search**

In [75]:
from PIL import Image

def generate_clip_image_embedding(image_path):
    """Generate CLIP embedding for an image query."""
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        image_embedding = clip_model.get_image_features(**inputs)
    return image_embedding.squeeze().cpu().numpy()

# Define an image query
query_image_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/test_image_renesas.png"
# Generate image embedding
query_image_embedding = generate_clip_image_embedding(query_image_path)

# Perform semantic search in ChromaDB using the image
search_results = image_text_collection.query(
    query_embeddings=[query_image_embedding.tolist()],
    n_results=5  # Retrieve top 5 results
)

# Display results
for i, (result, score) in enumerate(zip(search_results["metadatas"][0], search_results["distances"][0])):
    print(f"\n🔍 **Result {i+1}** (Similarity: {similarity:.4f})")
    print(f"📌 Model: {result['model_name']}")
    print(f"📜 Chunk: {result['chunk_text']}")
    print(f"🖼️ Image Path: {result['image_path']}")



🔍 **Result 1** (Similarity: 0.7844)
📌 Model: Bi-directional GaN DC/DC Converter for 12V/48V EV/HEV Systems
📜 Chunk: Bi-directional GaN DC/DC Converter for 12V/48V EV/HEV Systems. This is about application Bi-directional GaN DC/DC Converter for 12V/48V EV/HEV Systems: As electric vehicles (EVs) and hybrid electric vehicles (HEVs) gain popularity, there's a growing demand for efficient energy management and dual-voltage systems to transfer energy between 12V and 48V power nets. This design is ideal for 48V mild hybrid electric vehicles (MHEV) and electric motorcycles,featuringhighly efficient 12V/48V DC/DC converters. It reduces PCB area by 46% using Gallium Nitride (GaN) High-Electron-Mobility Transistors (HEMTs) with excellent switching characteristics, enabling high efficiency power conversion at a 500kHz switching frequency with small 1.3µH inductors. The system benefits for application Bi-directional GaN DC/DC Converter for 12V/48V EV/HEV Systems are: Bi-directional analog controll

#**Text + Image based input search**

In [73]:
def hybrid_embedding(text_embedding, image_embedding, text_weight=0.7, image_weight=0.3):
    """Combine text and image embeddings using weighted fusion."""
    if image_embedding is None:
        return text_embedding  # If no image is provided, use only text
    return (text_embedding * text_weight) + (image_embedding * image_weight)

# Generate embeddings for both text and image
query_text_embedding = generate_clip_text_embedding("which application/module does the attached image is related to? ")
query_image_embedding = generate_clip_image_embedding("/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/Test2.png")

# Combine embeddings
hybrid_query_embedding = hybrid_embedding(query_text_embedding, query_image_embedding)

# Perform search in ChromaDB
search_results = image_text_collection.query(
    query_embeddings=[hybrid_query_embedding.tolist()],
    n_results=5
)

# Display results
for i, (result, score) in enumerate(zip(search_results["metadatas"][0], search_results["distances"][0])):
    print(f"\n🔍 **Result {i+1}** (Similarity: {similarity:.4f})")
    print(f"📌 Model: {result['model_name']}")
    print(f"📜 Chunk: {result['chunk_text']}")
    print(f"🖼️ Image Path: {result['image_path']}")



🔍 **Result 1** (Similarity: 0.7844)
📌 Model: High-End Cockpit & Infotainment Solution
📜 Chunk: High-End Cockpit & Infotainment Solution. This is about application High-End Cockpit & Infotainment Solution: This combination of the R-Car (H3/M3/M3N) system-on-chip (SoC), power management IC (PMIC), and programmable clock generator allows for a versatile solution. They enable scalable cockpit and infotainment solutions that support high image quality, multiple video display outputs, and a wide variety of memory interfaces all in one design. The system benefits for application High-End Cockpit & Infotainment Solution are:  A versatile system that enables scalable cockpit and infotainment solutions that support high image quality, multiple video display outputs, and a wide variety of memory interfaces. Flexible clock generators can generate any clock frequency from 1MHz to 350MHz and allow a single device to replace several discrete clock circuits, saving BOM cost and reducing PCB area. A f

In [76]:
import chromadb

# Load the collection
image_text_collection = chroma_client.get_or_create_collection(name="rag_clip_image_text")

# Verify if data exists
stored_records = image_text_collection.get()
print(f"✅ Total Records after inserting into ChromaDB: {len(stored_records['ids'])}")

✅ Total Records after inserting into ChromaDB: 235


In [6]:
%%writefile app.py

import streamlit as st
import chromadb
import torch
from openai import OpenAI
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import os
from dotenv import load_dotenv

load_dotenv()

# Load OpenAI API Key
if "OPENAI_API_KEY" not in os.environ:
    st.error("❌ OpenAI API Key not found! Please set it in your .env file.")
    st.stop()

# Initialize OpenAI
client = OpenAI()

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_chromadb_store")
image_text_collection = chroma_client.get_collection(name="rag_clip_image_text")

# Load CLIP Model for Text & Image Embeddings
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Streamlit UI
st.title("🔍 RAG-Powered Chatbot (ChromaDB + GPT-4o Mini)")
st.write("Ask a question, and the system will search ChromaDB for relevant information before responding.")

# User Query Input
query_text = st.text_input("Enter your query:")
uploaded_image = st.file_uploader("Upload an image (optional)", type=["jpg", "png", "jpeg"])

def generate_clip_text_embedding(text):
    """Generate text embedding for user queries using CLIP."""
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True, max_length=77).to(device)
    with torch.no_grad():
        text_embedding = clip_model.get_text_features(**inputs)
    return text_embedding.squeeze().cpu().numpy()

def generate_clip_image_embedding(image):
    """Generate image embedding using CLIP."""
    image = Image.open(image).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        image_embedding = clip_model.get_image_features(**inputs)
    return image_embedding.squeeze().cpu().numpy()

if st.button("Search & Generate Answer"):
    if not query_text and not uploaded_image:
        st.warning("Please enter a query or upload an image.")
    else:
        # Encode the text query using CLIP
        query_embedding = generate_clip_text_embedding(query_text).tolist()

        # Process image query if uploaded
        image_embedding = None
        if uploaded_image:
            image_embedding = generate_clip_image_embedding(uploaded_image).tolist()

        # Hybrid Query (Text + Image)
        if image_embedding:
            hybrid_embedding = [(t + i) / 2 for t, i in zip(query_embedding, image_embedding)]
        else:
            hybrid_embedding = query_embedding

        # Perform ChromaDB Search
        search_results = image_text_collection.query(
            query_embeddings=[hybrid_embedding],  # Now using CLIP embeddings
            n_results=5
        )

                # Check if search results are empty
        if not search_results["metadatas"] or len(search_results["metadatas"][0]) == 0:
            st.error("⚠️ No relevant results found in ChromaDB!")
        else:
            # Retrieve context
            context = "\n".join([result["chunk_text"] for result in search_results["metadatas"][0]])


            # Prepare prompt for GPT-4o Mini
            prompt = [
                {"role": "system", "content": "You are a helpful AI assistant. Answer the following question based on the retrieved context."},
                {"role": "user", "content": f"Context: {context}\n\nQuestion: {query_text}"}
            ]

            # Query GPT-4o Mini
            completion = client.chat.completions.create(
                model="gpt-4o",
                messages=prompt
            )

            # Display GPT Response
            st.subheader("🤖 AI Response:")
            st.write(completion.choices[0].message.content)

            st.subheader("✅ Retrieved Context:")
            st.write(context)


Overwriting app.py


In [10]:
!pkill -f ngrok
!kill $(pgrep streamlit) 2>/dev/null

In [11]:
from pyngrok import ngrok
import time


# Start a fresh Ngrok tunnel (correct syntax)
public_url = ngrok.connect(addr="8501", proto="http")
print(f"🌍 Open your Streamlit app here: {public_url}")

# Run Streamlit in the background
!streamlit run app.py &>/dev/null &

🌍 Open your Streamlit app here: NgrokTunnel: "https://3db9-34-16-148-84.ngrok-free.app" -> "http://localhost:8501"
